# Picking

`ngsolve_webgpu` ships pick handlers that decode click events on rendered scenes back into mesh entities (elements, regions) or OCC geometry features (vertex/edge/face).

.. note::
   The pick handlers run **inside the live Jupyter session** — they need a Python kernel to receive the click events and react. The static scenes embedded on this page show what the renderers look like; to try picking interactively, run the notebook locally.

## API summary

| Class | Purpose |
|---|---|
| `MeshPickResult` | Decodes clicks on `MeshElements2d/3d` / `ClippingCF` renderers into `(element_id, region_index, kind)`. |
| `GeoPickResult` | Decodes clicks on `GeometryRenderer` into `(geo_type, index)` where `geo_type ∈ {0:vertex, 1:edge, 2:face}`. |
| `HighlightUniforms` | Uniform block (binding 57) shared by shaders to highlight the currently picked entity. |

## Mesh picking — example wiring

```python
from ngsolve_webgpu.pick import MeshPickResult, HighlightUniforms

highlight = HighlightUniforms()
scene = Draw(mesh)

def on_click(event):
    pick = MeshPickResult(event, mesh, options={}, kind="surface")
    print("clicked element", pick.element_id, "in region", pick.region_index)
    highlight.element_id = pick.element_id
    highlight.region_index = pick.region_index
    highlight.update_buffer()
    scene.redraw()

scene.on_select(on_click)
```

The static scene below is the canvas you would interact with:

In [ ]:
from ngsolve import *
from ngsolve_webgpu.jupyter import Draw

mesh = Mesh(unit_cube.GenerateMesh(maxh=0.3))
Draw(mesh)

## Geometry picking — OCC shapes

Drawing an `OCCGeometry` produces a `GeometryRenderer`. `GeoPickResult` decodes clicks into the underlying topological entity:

```python
from ngsolve_webgpu.pick import GeoPickResult

scene = Draw(shape)

def on_click(event):
    pick = GeoPickResult(event, geo, options={})
    kind = {0: "vertex", 1: "edge", 2: "face"}[pick.geo_type]
    print(f"picked {kind} #{pick.index}")

scene.on_select(on_click)
```

In [ ]:
from netgen.occ import Box, Pnt, OCCGeometry
from ngsolve_webgpu.jupyter import Draw

shape = Box(Pnt(0, 0, 0), Pnt(1, 1, 1))
Draw(OCCGeometry(shape))